# Семинар 13. Pandas / Polars

## Мотивация

Эксперимент закончился — на диске лежит выгрузка: сотня строк «запуск, модель,
датасет, learning rate, точность, сколько минут считали». Дальше у вас всегда
одни и те же вопросы: какая модель в среднем лучше, помог ли маленький lr, во
что обошлись по времени неудачные запуски, что показать научруку.

Написать это циклами по спискам можно, но получается длинно и хрупко: пропуск в
одной ячейке — и среднее уже неверное, добавили колонку — и переписывать надо
половину. **pandas** даёт для таких данных один объект — таблицу `DataFrame` — и
набор операций над ней целиком: отобрать, сгруппировать, склеить с другой
таблицей, отсортировать. Ровно тот набор, который в семинаре 12 давал SQL, но
результат остаётся объектом Python: его можно посчитать дальше, нарисовать
графиком и сохранить.

Сюжет занятия — не список функций, а одна задача: **ответить на вопросы по
выгрузке экспериментов**. По дороге разберём три места, где новички чаще всего
получают тихо неверный ответ: путаница `loc`/`iloc`, присваивание через цепочку
(`chained assignment`) и пропуски, которые «заменили на ноль».

В конце — **Polars**: та же таблица, но другой язык запросов (выражения) и
ленивое исполнение. Он нужен не всегда, и понимать, когда именно, полезнее, чем
знать его API.

Проверено на pandas 3.0.5 и polars 1.43.2 — версии печатаются в первых
ячейках, сверьтесь. На pandas 2 часть примеров ведёт себя иначе, об этом будет
сказано по ходу. Домашка — мини-исследование датасета
в pandas, так что всё, что здесь есть, пригодится сразу.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

pandas написал Уэс Маккинни в 2008 году, работая в хедж-фонде AQR: ему нужно было
считать «панельные данные» (panel data — отсюда и имя), а в Python для этого не было
ничего. В 2009-м библиотеку открыли, и она стала стандартом де-факто: почти любой
пример анализа данных на Python начинается со строчки `import pandas as pd`.

Polars моложе на двенадцать лет: Ритчи Винк начал его в 2020-м, на Rust и поверх
колоночного формата Apache Arrow. По духу это «а что если написать pandas заново,
зная всё, что мы про него теперь знаем»: отсюда и отсутствие индекса, и ленивое
исполнение — обе вещи в pandas уже не встроить, не сломав совместимость с миллионом
написанных ноутбуков.

</details>

## 1. Таблица: DataFrame и Series

`DataFrame` — двумерная таблица: **именованные столбцы** и **индекс** строк.
Столбец — это `Series`: одномерный массив значений плюс тот же индекс. Всё
остальное в pandas — операции над этими двумя объектами.

Соберём маленькую выгрузку из 12 запусков обучения, чтобы всё было видно
глазами.

In [ ]:
import pandas as pd

print(pd.__version__)   # сверьтесь с версией из мотивации: на pandas 2 часть примеров ведёт себя иначе

Таблицу собирают из словаря: ключ — имя столбца, значение — список значений
этого столбца. Начнём с двух столбцов, остальные допишем следующей ячейкой.

In [ ]:
runs = pd.DataFrame({
    "run_id": [f"r{i:02d}" for i in range(1, 13)],   # r01..r12 — имена запусков
    "model": ["resnet18"] * 3 + ["resnet50"] * 3 + ["vit_small"] * 3 + ["mobilenet_v3"] * 3,
})
print(runs.head(3))     # head(3) — первые три строки; слева появится индекс 0, 1, 2

Слева без заголовка — **индекс**: по умолчанию это номера строк `0..11`.
Индекс не то же самое, что позиция; сейчас увидим, чем он полезен, а
чуть позже — чем опасен.

Новый столбец добавляется присваиванием: справа список нужной длины.
Пустое значение записываем как `None` — это будет пропуск, а не ноль.

In [ ]:
runs["dataset"] = ["cifar10", "cifar10", "imagenette"] * 4   # присваивание по новому имени = новый столбец
runs["lr"] = [0.1, 0.01, 0.001] * 4
runs["val_acc"] = [0.872, 0.901, 0.815, 0.915, 0.908, None, 0.924, 0.931, 0.860, 0.848, 0.859, 0.771]   # None у r06: обучение разошлось
runs["train_min"] = [12.5, 12.8, 25.0, 31.2, 30.9, 29.7, 44.0, 43.5, 88.0, 8.0, 8.2, 16.0]
runs["gpu"] = ["a100"] * 6 + ["h100"] * 3 + ["a100", None, "a100"]   # у r11 имя GPU забыли записать
print(runs.columns.tolist())   # семь столбцов вместо двух

Пять присваиваний — пять новых столбцов, выгрузка готова. Первое, что делают с
новой таблицей: смотрят её размер и **типы столбцов**.

In [ ]:
print(runs.shape)     # (строк, столбцов) — быстрый ответ «сколько всего данных»
print(runs.dtypes)    # тип у столбца один на все строки сразу

Анатомия DataFrame

На схеме та же таблица, только покороче: имена столбцов сверху, типы под ними,
индекс слева, а один столбец вынут отдельно — это и есть `Series`.

#### ❓ **Вопрос**: Почему у `lr`, `val_acc` и `train_min` тип `float64`, у `model` — `str`, а у `run_id` не какой-нибудь «идентификатор»?

<details>

<summary><strong>Ответ</strong></summary>

Тип столбца pandas выводит из данных, и он **один на весь столбец**: в `val_acc` лежат дробные числа, поэтому `float64`; в `model` и `run_id` — текст, поэтому `str`. Никакого «типа идентификатора» нет: `run_id` для pandas просто строки, смысл ключа появляется только тогда, когда мы сами используем столбец в `merge`.

Полезно знать: `str` как отдельный тип столбца — это pandas 3. В pandas 2 и старее та же таблица показала бы `object`, и примеры из интернета часто написаны именно про `object`.

</details>

### Зачем нужен индекс

Номера строк `0..11` кажутся бесполезными, но индексом можно сделать
осмысленный ключ — и обращаться к таблице по нему, а не по номеру.

In [ ]:
by_id = runs.set_index("run_id")     # столбец run_id стал индексом: метка строки — её имя
print(by_id.loc["r05", "val_acc"])   # loc[метка строки, имя столбца] — точечное обращение
print(by_id.loc[["r02", "r07"], "model"].tolist())   # список меток — несколько строк сразу

Второе свойство важнее: по индексу pandas **выравнивает** данные. В
арифметике двух `Series` складываются значения с одинаковой меткой, а не
стоящие на одной позиции. Проверим: вычтем из столбца его же копию,
перевёрнутую задом наперёд (берём четыре значения, чтобы перестановка
задела каждое). `iloc` ниже — обращение по номеру строки, срез `[::-1]`
переворачивает порядок; подробно про `iloc` будет в разделе 2.

In [ ]:
first = by_id["val_acc"].head(4)   # первые четыре запуска, метки r01..r04
reversed_ = first.iloc[::-1]       # тот же Series задом наперёд: метки переехали вместе со значениями
print(reversed_.index.tolist())    # порядок меток стал r04, r03, r02, r01

Порядок строк во втором `Series` обратный. Теперь вычтем один из другого — и
посмотрим, какие значения pandas посчитал соответствующими друг другу.

In [ ]:
print((first - reversed_).tolist())   # четыре нуля: вычиталось по меткам, а не по позициям

#### ❓ **Вопрос**: Разность вышла нулевой, хотя во втором `Series` значения идут в обратном порядке. А что получится, если у второго `Series` метки будут другие — скажем, `r03..r06`?

<details>

<summary><strong>Ответ</strong></summary>

Ноль — потому что pandas вычитает не «первое из первого», а «`r01` из `r01`»: перед операцией он выравнивает оба `Series` по индексу, и порядок строк на результат не влияет.

А вот с метками `r03..r06` результат будет длиной в **объединение** меток — шесть строк `r01..r06`. Там, где метка есть только у одного из двух `Series` (`r01`, `r02`, `r05`, `r06`), получится `NaN`, и никакой ошибки не будет. Это и есть главная ловушка выравнивания: несовпадение ключей превращается не в исключение, а в тихие пропуски.

```
first      r01 0.872   r02 0.901   r03 0.815   r04 0.915
reversed_  r04 0.915   r03 0.815   r02 0.901   r01 0.872

pandas складывает по меткам:   r01↔r01   r02↔r02   r03↔r03   r04↔r04
numpy складывал бы по местам:  1↔1       2↔2       3↔3       4↔4
```

Со списками или массивами numpy вычитание пошло бы по позициям, и все четыре разности были бы ненулевыми: `0.872 − 0.915`, `0.901 − 0.815` и так далее.

Поэтому индекс — одновременно и удобство (`loc` по ключу, выравнивание при склейке), и источник тихих сюрпризов.

</details>

### Файл на диске: read_csv и to_csv

Обычно таблицу не собирают руками, а читают из csv. Сохраним нашу и посмотрим,
что оказалось в файле.

In [ ]:
from pathlib import Path

work = Path.home() / "seminar-13"   # работаем в домашнем каталоге: /tmp вычищается при перезагрузке
work.mkdir(exist_ok=True)           # exist_ok=True — повторный запуск ячейки не упадёт
print(work)

Каталог создан один раз, дальше все файлы кладём в него. Сохраним таблицу в csv
«как обычно» — без единого параметра — и заглянем в первую строку файла.

In [ ]:
runs.to_csv(work / "runs-bad.csv")   # запись со всеми умолчаниями
header = (work / "runs-bad.csv").read_text(encoding="utf-8").splitlines()[0]
print(header)   # заголовок начинается с запятой: у первого столбца нет имени

Прочитаем этот файл обратно и посмотрим на имена столбцов.

In [ ]:
back = pd.read_csv(work / "runs-bad.csv")
print(back.columns.tolist())   # столбцов на один больше, чем было при записи

#### ❓ **Вопрос**: Откуда в прочитанной таблице взялся столбец `Unnamed: 0`, которого не было в исходной?

<details>

<summary><strong>Ответ</strong></summary>

`to_csv` по умолчанию пишет **индекс** первым столбцом, и в заголовке у него нет имени — это видно по первой строке файла: она начинается с запятой. При чтении pandas обязан как-то назвать безымянный столбец и называет его `Unnamed: 0`.

Лечится на записи: `to_csv(..., index=False)`. Если файл уже такой — `read_csv(..., index_col=0)`, тогда первый столбец станет индексом, а не данными. Классический симптом: после нескольких «сохранил — прочитал» в таблице копятся `Unnamed: 0`, `Unnamed: 0.1`, …

</details>

In [ ]:
runs.to_csv(work / "runs.csv", index=False)   # index=False — не писать индекс в файл
runs = pd.read_csv(work / "runs.csv")         # дальше работаем с прочитанным, как с настоящей выгрузкой
print(runs.columns.tolist())                  # никаких Unnamed: 0
print(runs.shape)

Дальше работаем именно с прочитанной из файла таблицей — как было бы с
настоящей выгрузкой.

У `read_csv` больше полусотни параметров; помнить надо четыре:

- `sep=";"` — если разделитель не запятая (типичная беда выгрузок из Excel);
- `encoding="cp1251"` — если файл не в UTF-8. Кодировки разбирались в семинаре
  10, здесь просто помните, что параметр есть и его иногда надо указать;
- `nrows=1000` — прочитать кусок большого файла, чтобы посмотреть на структуру;
- `na_values=["-", "n/a"]` — что ещё считать пропуском кроме пустого поля.

И отдельно: csv **нельзя** разбирать через `split(",")`. В файле бывают
экранированные значения с запятой внутри кавычек — `read_csv` их понимает, а
`split` порежет строку в неверном месте.

Первые два параметра стоит увидеть в деле: сделаем файл с точкой с запятой
и прочерком вместо пустого поля — ровно то, что приезжает из Excel.

In [ ]:
raw = "run_id;val_acc\nr20;0.83\nr21;-\n"   # поля разделены «;», пустое значение записано прочерком
(work / "excel.csv").write_text(raw, encoding="utf-8")
print(raw)   # вот что теперь лежит в файле

Файл готов. Читаем его, объяснив pandas обе особенности сразу: `sep=";"` — чем
разделены поля, `na_values=["-"]` — что считать пропуском.

In [ ]:
other = pd.read_csv(work / "excel.csv", sep=";", na_values=["-"])
print(other)   # прочерк превратился в NaN
print(other["val_acc"].dtype, other["val_acc"].isna().sum())   # столбец остался числовым, пропуск ровно один

Без `sep=";"` вся строка попала бы в один столбец, без `na_values=["-"]`
прочерк остался бы текстом — и весь столбец стал бы `str`, а не `float64`,
после чего `mean()` перестал бы работать.

Оставшиеся два параметра — на файле в cp1251: `encoding` говорит, в какой
кодировке читать, `nrows` — сколько строк взять, чтобы просто посмотреть на
структуру большого файла.

In [ ]:
text = "run_id,note\nr20,Разошлось на первой эпохе\nr21,Всё хорошо\n"
(work / "cp1251.csv").write_bytes(text.encode("cp1251"))   # кладём файл не в UTF-8, а в cp1251
print((work / "cp1251.csv").stat().st_size, "байт")        # в cp1251 кириллическая буква занимает один байт

Файл записан в чужой кодировке. Читаем — и просим только первую строку данных.

In [ ]:
print(pd.read_csv(work / "cp1251.csv", encoding="cp1251", nrows=1))   # encoding — как декодировать, nrows — сколько строк взять

Без `encoding="cp1251"` чтение упало бы с `UnicodeDecodeError`; `nrows=1`
вернул ровно одну строку данных. Кодировки подробно разбирались в семинаре
10.

Когда файл прочитан, следующий шаг — обзорная сводка по числовым столбцам.

In [ ]:
print(runs.describe())   # count, mean, std, min, квартили, max по каждому числовому столбцу

#### ❓ **Вопрос**: У `lr` и `train_min` в строке `count` стоит 12, а у `val_acc` — 11. Значит ли это, что таблица «рваная»?

<details>

<summary><strong>Ответ</strong></summary>

Нет, строк по-прежнему 12: `runs.shape` показал `(12, 7)`. `count` — это число **непропущенных** значений в столбце, а у нас один запуск (`r06`) остался без `val_acc`: обучение разошлось, метрику писать было нечего.

Отсюда практическое правило: `describe()` смотрят не ради красивых средних, а чтобы поймать несостыковки — разные `count` по столбцам, подозрительные `min`/`max`, нулевое стандартное отклонение. `lr` в этой сводке, кстати, усреднять бессмысленно: это категория, записанная числом.

</details>

## 2. Отбор строк и столбцов

Один столбец в квадратных скобках — `Series`, список столбцов — `DataFrame`.
Разница видна по типу и по тому, как объект печатается.

In [ ]:
acc = runs["val_acc"]     # одна пара скобок — Series
print(type(acc).__name__, type(runs[["val_acc"]]).__name__)   # две пары скобок — DataFrame из одного столбца
print(acc.head(3))

#### ❓ **Вопрос**: Зачем различать `runs["val_acc"]` и `runs[["val_acc"]]`, если данные в них одни и те же?

<details>

<summary><strong>Ответ</strong></summary>

У них разные типы: первый — `Series`, второй — `DataFrame` из одного столбца. От типа зависит, что можно сделать дальше: у `Series` `.mean()` возвращает одно число, а у `DataFrame` — `Series` со средним по каждому столбцу (индекс — имена столбцов), даже если столбец один. И наоборот: `merge` — метод таблицы, у `Series` его просто нет; а `describe` и `to_csv` есть у обоих, только у `Series` они описывают и сохраняют один столбец.

Практически: `Series` — для вычислений над одним столбцом, двойные скобки — когда нужен именно кусок таблицы, например `runs[["run_id", "val_acc"]]` на печать или на запись в csv.

</details>

Строки отбирают **маской**: сравнение столбца с числом даёт `Series` из
`True`/`False` той же длины, а таблица в квадратных скобках оставляет строки,
где `True`.

In [ ]:
mask = runs["val_acc"] > 0.9   # сравнение столбца с числом даёт Series из True/False длиной в таблицу
print(mask.head(3).tolist())   # по одному ответу на строку
print(runs[mask].shape)        # таблица в квадратных скобках с маской = только строки, где True

Условия объединяются через `&` (и), `|` (или), `~` (не), и **каждое надо
брать в скобки**: у `&` приоритет выше, чем у сравнения. Питоновские `and`/`or`
здесь не работают — они требуют от аргумента одного `True`/`False`, а у нас
массив.

In [ ]:
slow = runs[(runs["train_min"] > 40) | (runs["val_acc"].isna())]   # | = «или»: долгие ЛИБО без метрики
print(len(slow))

Четыре запуска: три считались дольше сорока минут, а четвёртый разошёлся и метрики
не имеет. Теперь отрицание — «всё, кроме».

In [ ]:
others = runs[~(runs["model"] == "resnet18")]   # ~ = «не»: все модели, кроме resnet18
print(others["model"].unique().tolist())        # unique() — какие значения остались

Скобки вокруг каждого сравнения обязательны: без них `~ runs["model"] == "resnet18"`
разобралось бы как `(~runs["model"]) == "resnet18"` — то есть совсем не то, что мы
имели в виду. Условия можно комбинировать и дальше: `&` — «и».

In [ ]:
best = runs[(runs["val_acc"] > 0.9) & (runs["dataset"] == "cifar10")]   # & = «и»: оба условия сразу
print(best[["run_id", "model", "val_acc"]])   # список столбцов в двойных скобках — таблица из трёх столбцов

### loc против iloc

`loc` обращается **по метке индекса**, `iloc` — **по позиции**. Пока индекс —
это `0..N`, разницы не видно. Она появляется сразу после первой же фильтрации:
у отобранных строк метки сохраняются, а позиции пересчитываются.

In [ ]:
good = runs[runs["val_acc"] > 0.9]   # отобрали лучшие запуски
print(good.index.tolist())           # метки достались от исходной таблицы, нумерация с нуля не началась

Позиции в отобранном куске считаются заново с нуля, а метки остались прежними.
Отсюда все различия `iloc` и `loc`.

In [ ]:
print(good.iloc[0]["run_id"])   # iloc[0] — первая по счёту строка того, что осталось
print(good.loc[1, "run_id"])    # loc[1] — строка с меткой 1, здесь это та же самая строка

In [ ]:
try:
    print(good.loc[0])       # метки 0 в отобранном куске нет: r01 не прошёл фильтр
except KeyError as exc:
    print("KeyError:", exc)

#### ❓ **Вопрос**: `good.iloc[0]` вернул строку `r02`, `good.loc[1]` — тоже `r02`, а `good.loc[0]` упал с `KeyError`. Почему?

<details>

<summary><strong>Ответ</strong></summary>

После фильтрации индекс отобранных строк — `[1, 3, 4, 6, 7]`, метки исходной таблицы сохранились. `iloc[0]` — «нулевая по счёту строка того, что осталось», это `r02`. `loc[1]` — «строка с меткой 1», это тот же `r02`. А метки `0` в отобранном куске нет — строка `r01` не прошла фильтр, отсюда `KeyError`.

Вывод: после фильтрации `loc` и `iloc` дают разное, и `loc` может упасть. Если позиции нужны заново с нуля — `good.reset_index(drop=True)`.

</details>

### Ловушка: присваивание через цепочку

Вот как писать **не надо** — «отобрать, а потом записать» в два
обращения: `df[маска][столбец] = ...`. Попробуем так поднять до 0.8 все
точности ниже 0.8.

Эксперимент ставим на копии: `runs.copy()` — независимая таблица, и что бы
мы с ней ни сделали, `runs` для следующих ячеек останется прежней.

In [ ]:
bad = runs.copy()                            # эксперимент на копии, исходную таблицу не портим
bad[bad["val_acc"] < 0.8]["val_acc"] = 0.8   # так писать НЕ надо: два обращения подряд
print(bad["val_acc"].min())                  # минимум не изменился — запись потерялась

Минимум остался прежним: **запись не дошла до таблицы**. Ячейка при этом не
упала: `ChainedAssignmentError` здесь — предупреждение, а не исключение,
`min()` спокойно напечатался. Тем и опасно: в скрипте, где предупреждения
никто не читает, ошибка пройдёт незамеченной.

Правильный способ — одно обращение `loc[строки, столбец]`.

In [ ]:
fixed = runs.copy()
fixed.loc[fixed["val_acc"] < 0.8, "val_acc"] = 0.8   # одно обращение: строки и столбец в одном loc
print(fixed["val_acc"].min())                        # 0.8 — запись дошла до таблицы

#### ❓ **Вопрос**: Почему присваивание в две квадратные скобки ничего не изменило, хотя ошибки-исключения не было?

<details>

<summary><strong>Ответ</strong></summary>

`bad[bad["val_acc"] < 0.8]` — это **новый временный объект**, копия отобранных строк. Запись `["val_acc"] = 0.8` меняет его, после чего он тут же удаляется, а исходная таблица остаётся нетронутой — это и показал `min()`, оставшийся 0.771.

В `fixed.loc[маска, "val_acc"] = 0.8` отбор строк и выбор столбца происходят в **одном** обращении, поэтому pandas пишет в саму таблицу, и минимум стал 0.8.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

До pandas 3 такой код иногда всё-таки срабатывал: результат зависел от того, легла ли
отобранная часть в память отдельной копией или осталась видом на исходные данные.
Предупреждение об этом — `SettingWithCopyWarning` — было самым знаменитым сообщением
библиотеки: по нему написаны сотни ответов на Stack Overflow, и добрая половина из них
советовала «просто допишите `.copy()`», не объясняя, почему.

В pandas 3 включён copy-on-write: поведение стало предсказуемым (не работает никогда), а
`SettingWithCopyWarning` из библиотеки убрали совсем. Если встретите его в чужом коде или
в ответе чат-бота — перед вами пример из старой версии.

</details>

## 3. Пропуски: NaN — это не ноль

Пропуск (`NaN`, not a number) означает «значение неизвестно». Ноль означает
«известно, и оно равно нулю». Смешивать их нельзя: запуск, который разошёлся,
не имеет точности 0 % — у него точности нет вообще.

Технически `NaN` — значение типа `float`. Отсюда неожиданный побочный
эффект: столбец целых чисел, в котором есть хоть один пропуск, приезжает
из csv уже дробным.

In [ ]:
print(pd.Series([1, 2, 3]).dtype)      # int64: целые без пропусков
print(pd.Series([1, None, 3]).dtype)   # float64: из-за одного None весь столбец стал дробным

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

`NaN` придумали не в pandas: это часть стандарта IEEE 754 для чисел с плавающей точкой —
того же, что описывает бесконечности. У него есть свойство, которое сначала кажется
ошибкой: `float("nan") != float("nan")` истинно. «Неизвестно» не равно «неизвестно» —
и это логично, две неизвестные величины совпадать не обязаны.

Практическое следствие: сравнение `runs["val_acc"] == float("nan")` не найдёт ни одного
пропуска, и никакой ошибки при этом не будет. Поэтому для пропусков есть отдельный метод
`isna()`, и только им их и проверяют.

</details>

Поэтому «почему у меня число эпох стало `10.0`» — почти всегда значит «в
столбце есть пропуск». Сколько их и где — считают так.

In [ ]:
print(runs.isna().sum())   # по каждому столбцу: сколько в нём пропусков

`isna()` даёт таблицу из `True`/`False`, `sum()` складывает её по столбцам
(`True` считается за единицу).

Посмотреть, что вообще лежит в столбце, помогает `value_counts`: он
считает, сколько раз встретилось каждое значение. Пропуски он по умолчанию
не показывает — их надо запросить явно.

In [ ]:
print(runs["gpu"].value_counts(dropna=False))   # dropna=False — показать и строку NaN

Без `dropna=False` строки `NaN` в выводе просто не было бы, и столбец
выглядел бы полным. Теперь посмотрим, что делает «заменим пропуски на
ноль».

In [ ]:
print(runs["val_acc"].mean())               # среднее по непропущенным значениям
print(runs["val_acc"].fillna(0).mean())     # тот же расчёт, но пропуск объявлен нулём
print(runs["val_acc"].count(), len(runs))   # count — непропущенных значений, len — всего строк

#### ❓ **Вопрос**: Почему среднее упало с 0.8731 до 0.8003 и какое из двух значений отвечает на вопрос «какая у нас типичная точность»?

<details>

<summary><strong>Ответ</strong></summary>

Арифметика pandas по умолчанию **пропускает** `NaN`: `mean()` сложил 11 значений и поделил на 11 — это видно по `count() == 11` при `len() == 12`. После `fillna(0)` пропуск превратился в честный ноль: сумма та же (9.604), а делить стали на 12, отсюда 9.604 / 12 = 0.8003.

На вопрос отвечает первое число: 0.8731 — типичная точность **среди доехавших до конца запусков**. Второе не отвечает ни на какой вопрос: оно смешало «неизвестно» с «ноль». `fillna(0)` уместен там, где ноль — осмысленное значение по умолчанию (например, «сколько раз перезапускали»), а не как способ убрать `NaN` с глаз долой.

</details>

Второй способ — выбросить строки с пропусками, `dropna()`. Здесь легко
потерять больше, чем собирались.

In [ ]:
print(runs.dropna().shape)                     # выбрасывает строку, если пропуск хоть в одном столбце
print(runs.dropna(subset=["val_acc"]).shape)   # subset — смотреть только на нужный столбец

#### ❓ **Вопрос**: `dropna()` оставил 10 строк, `dropna(subset=["val_acc"])` — 11. Какая строка потерялась в первом случае и почему это плохо?

<details>

<summary><strong>Ответ</strong></summary>

`dropna()` без аргументов выбрасывает строку, если пропуск есть **хоть в одном** столбце. Кроме запуска без `val_acc` под нож попал `r11`, у которого не записано имя GPU (в `isna().sum()` видно: пропуски есть в двух столбцах).

Плохо это тем, что запуск с прекрасной точностью выпал из анализа из-за постороннего столбца, и молча. Поэтому `subset` указывают почти всегда: выбрасываем строки, где нет **того, что мы считаем**.

</details>

## 4. Группировки: groupby + agg

Вопрос «какая модель в среднем лучше» — это «разбей строки на группы по
значению столбца и посчитай в каждой». В pandas — `groupby`.

groupby: разделить, посчитать, склеить

Три кадра — три части одной команды: `groupby` только режет таблицу на группы,
считает уже `agg`, а результат склеивается обратно, по одной строке на группу.

In [ ]:
print(runs.groupby("model")["val_acc"].mean())   # средняя точность внутри каждой модели

Результат — `Series`, у которой индексом стали названия моделей (группы
отсортированы по ключу). Считать в группе можно сразу несколько функций —
`agg` со списком.

In [ ]:
print(runs.groupby("model")["val_acc"].agg(["size", "count", "mean"]))   # три функции сразу — три столбца в ответе

Ключом может быть и несколько столбцов сразу — тогда в `groupby` передают
список, а в результате индекс становится двухуровневым.

In [ ]:
print(runs.groupby(["dataset", "model"])["val_acc"].mean().head(4))   # ключ из двух столбцов — двухуровневый индекс

#### ❓ **Вопрос**: У `resnet50` в столбце `size` стоит 3, а в `count` — 2. В чём разница между этими двумя «счётчиками»?

<details>

<summary><strong>Ответ</strong></summary>

`size` — сколько **строк** попало в группу, включая пропуски. `count` — сколько в группе **непропущенных значений** того столбца, который мы выбрали (`val_acc`). У `resnet50` три запуска, но один (`r06`) разошёлся и точности не имеет.

Практический смысл: `size` отвечает «сколько раз запускали», `count` — «по скольким запускам посчитано среднее». Если в отчёте они разошлись — в данных есть дырки, и это надо показать, а не прятать.

</details>

Когда нужны разные функции по разным столбцам и человеческие имена
результата — именованная агрегация: `имя=(столбец, функция)`. У функции `size`
столбец в паре ни на что не влияет — она считает строки группы, а не значения,
поэтому там годится любое имя (в примере ниже — `run_id`).

In [ ]:
report = runs.groupby("model").agg(   # именованная агрегация: имя=(столбец, функция)
    n=("run_id", "size"),             # size считает строки группы, столбец здесь не важен
    best=("val_acc", "max"),          # лучшая точность модели
    total_min=("train_min", "sum"),   # сколько всего минут на неё потратили
)
print(report)

Ключ группировки ушёл в индекс — поэтому `model` в выводе стоит отдельной
строкой над заголовками. Вернуть его в обычный столбец (например, чтобы
сохранить в csv или сделать `merge`) — `reset_index()`.

In [ ]:
print(report.index.name)                       # ключ группировки лежит в индексе, а не в столбце
print(report.reset_index().columns.tolist())   # reset_index() возвращает его обычным столбцом

Сводка готова — а раз она осталась обычным объектом Python, её можно тут же
нарисовать: у `DataFrame` и `Series` есть `.plot`, поверх matplotlib из
семинара 1. Это то, что понадобится в домашке.


In [ ]:
import matplotlib.pyplot as plt

ax = report["best"].plot.bar(title="Лучшая val_acc по моделям", rot=15)   # индекс станет подписями столбиков
ax.set_ylabel("val_acc")
ax.set_ylim(0.75, 0.95)   # без обрезки оси разница между моделями не читается
plt.tight_layout()        # чтобы подписи не обрезались по краям

Одна строка вместо цикла с подписями осей: `.plot.bar()` берёт индекс за
подписи столбиков, а значения — за высоту. Дальше — обычный matplotlib:
возвращается его `Axes`, и всё, что вы про него знаете, работает.


<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Если помните семинар 12: `runs.groupby("model").agg(n=("run_id", "size"))` — это в
точности `SELECT model, count(*) AS n FROM runs GROUP BY model`. Модель одна и та же:
разбить на группы, свернуть каждую в одну строку.

Совпадение не случайное: и SQL, и pandas выросли из работы с реляционными таблицами, а
`groupby` в pandas прямо описан в документации через split-apply-combine — ту же схему,
что стоит за `GROUP BY`. Отличие в том, что здесь результат — объект Python, к которому
можно применить следующую операцию, не выходя из программы и не сериализуя ничего в
строку запроса.

</details>

## 5. Склейка таблиц: merge

Про сами модели в выгрузке ничего нет — семейство и размер лежат в отдельном
справочнике. Заведём его. Обратите внимание: справочник неполный (нет
`mobilenet_v3`) и содержит лишнее (`efficientnet_b0` мы не запускали) — так и
бывает с чужими таблицами.

In [ ]:
models = pd.DataFrame({
    "model": ["resnet18", "resnet50", "vit_small", "efficientnet_b0"],   # efficientnet_b0 мы не запускали
    "family": ["cnn", "cnn", "transformer", "cnn"],
    "params_m": [11.7, 25.6, 22.0, 5.3],
})
print(models)   # mobilenet_v3 из нашей выгрузки здесь отсутствует

Справочник понадобится ещё раз в разделе про polars — сохраним его рядом с `runs.csv`.

In [ ]:
models.to_csv(work / "models.csv", index=False)      # index=False — как договорились
print(sorted(p.name for p in work.glob("*.csv")))    # что уже лежит в рабочем каталоге

Склейка — это `merge`: «к каждой строке слева подставь то, что нашлось справа по
ключу `model`».

In [ ]:
joined = runs.merge(models, on="model", how="left")   # how="left" — сохранить все строки выгрузки
print(joined.shape)                                   # столбцов стало больше, строк — столько же
print(joined[["run_id", "model", "family"]].tail(3))  # в хвосте mobilenet_v3, которому справочник ничего не дал

#### ❓ **Вопрос**: После `merge` строк осталось 12, но у запусков `mobilenet_v3` в `family` стоит `NaN`. Что означает `how="left"` и когда он предпочтительнее `how="inner"`?

<details>

<summary><strong>Ответ</strong></summary>

`how="left"` — «сохрани все строки левой таблицы, подставь совпадения из правой, где их нет — поставь пропуск». Поэтому число строк не изменилось (12), а у моделей, которых нет в справочнике, новые столбцы пусты.

`how="inner"` оставил бы только совпавшие строки — три запуска `mobilenet_v3` просто исчезли бы, и таблица стала бы короче. Для анализа это опаснее: потеря строк не заметна, если не проверять размер. Правило простое — берите `how="left"` и после склейки смотрите `joined["family"].isna().sum()`: это прямой ответ на вопрос «для скольких строк справочник не сработал».

</details>

Проверять склейку глазами не надо: `indicator=True` добавляет к результату
служебный столбец `_merge` со значениями `left_only`, `right_only`, `both`.

In [ ]:
chk = runs.merge(models, on="model", how="left", indicator=True)   # indicator=True добавит столбец _merge
print(chk["_merge"].value_counts())   # отчёт о склейке в трёх числах

`both 9` — девяти запускам справочник что-то дал, `left_only 3` — трём не дал
ничего. `right_only` при `how="left"` всегда `0`: лишние строки справочника в
результат просто не попадают.

Одноимённый ключ бывает не всегда. Если столбцы называются по-разному —
`left_on`/`right_on`; если ключ составной — список: `on=["model", "dataset"]`.

In [ ]:
alias = models.rename(columns={"model": "model_name"})   # у справочника ключ теперь называется иначе
byname = runs.merge(alias, left_on="model", right_on="model_name", how="left")   # слева одно имя, справа другое
print(byname.columns.tolist())   # оба столбца-ключа остались в результате

Составной ключ — это список в `on`: строки считаются совпавшими, только
если совпали **оба** столбца. Так подтягивают, например, целевую точность
для пары «модель + датасет».

In [ ]:
targets = pd.DataFrame({          # целевая точность своя для каждой пары «модель + датасет»
    "model": ["resnet18", "resnet18"],
    "dataset": ["cifar10", "imagenette"],
    "target": [0.90, 0.83],
})
print(targets)

Склеиваем по паре столбцов сразу: строка считается совпавшей, только если сошлись
**оба** значения.

In [ ]:
with_target = runs.merge(targets, on=["model", "dataset"], how="left")   # список в on — составной ключ
print(with_target.loc[with_target["target"].notna(), ["run_id", "dataset", "target"]])   # notna() — только те, кому цель нашлась

А если в обеих таблицах есть **одноимённый неключевой** столбец, pandas
разведёт их суффиксами. Свои суффиксы задаются параметром `suffixes` —
сравним наши точности с эталонными.

In [ ]:
baseline = pd.DataFrame({"model": ["resnet18", "resnet50"], "val_acc": [0.85, 0.90]})   # столбец val_acc есть и слева, и справа
print(baseline)

Ключ `model` в обеих таблицах — это нормально, он один на двоих. А вот `val_acc`
встречается с обеих сторон и ключом не является: pandas обязан их как-то развести.

In [ ]:
cmp = runs.merge(baseline, on="model", how="left", suffixes=("", "_baseline"))   # свои суффиксы вместо _x/_y
print(cmp[["run_id", "val_acc", "val_acc_baseline"]].head(3))   # наши точности рядом с эталонными

Без `suffixes` те же столбцы получили бы имена по умолчанию — `val_acc_x` и
`val_acc_y`, и через две строки кода уже не вспомнить, где чьё.

Остались `inner` и `outer` — они отличаются как раз судьбой несовпавших строк.

Виды merge: left, inner, outer

Проверим схему на наших таблицах: сначала `inner`.

In [ ]:
print(runs.merge(models, on="model", how="inner").shape)   # inner: остались только совпавшие строки

Три запуска исчезли, и никакого предупреждения при этом не было. Теперь `outer`.

In [ ]:
outer = runs.merge(models, on="model", how="outer")   # outer: и все левые, и все правые
print(outer.shape)
print(outer[["run_id", "model", "val_acc", "family"]].head(3))   # первой идёт строка справочника без запуска

`inner` дал 9 строк (потеряли три запуска), `outer` — 13: к двенадцати
запускам добавилась строка про `efficientnet_b0` — она в выводе первая, и в
ней пусто всё, что приходит слева (`run_id`, `val_acc`). А у `mobilenet_v3`
наоборот пусто справа. Итого:

| `how` | какие строки остаются | где склейка добавит `NaN` | строк у нас |
|---|---|---|---|
| `left` | все левые | в столбцах справочника | 12 |
| `inner` | только совпавшие | нигде | 9 |
| `outer` | все левые и все правые | с обеих сторон | 13 |

Речь о **новых** пропусках, которые порождает сама склейка: те, что были в
данных раньше (пустой `val_acc` у `r06`), никуда не денутся ни при каком
`how`.

Теперь посчитаем среднюю точность по семействам.

In [ ]:
print(joined.groupby("family")["val_acc"].count())                 # сколько точностей посчитано в каждом семействе
print(joined.groupby("family", dropna=False)["val_acc"].count())   # dropna=False — не терять строки с пропуском в ключе

#### ❓ **Вопрос**: В первом выводе две группы, и в сумме по ним 5 + 3 = 8 запусков вместо 12. Куда делись остальные четыре?

<details>

<summary><strong>Ответ</strong></summary>

Потери двух разных сортов, и это ровно те грабли, о которых шла речь выше.

1. Три строки `mobilenet_v3` исчезли целиком: `groupby` по умолчанию **выбрасывает строки, у которых ключ группировки — пропуск**, а `family` у них пуст после `merge` с неполным справочником. `dropna=False` во втором выводе возвращает их отдельной группой `NaN`.
2. Ещё одна строка — разошедшийся `resnet50` (`r06`): она в группе `cnn` есть, но `count` считает непропущенные значения `val_acc`, а его нет.

Опасность в том, что и то и другое происходит молча. Единственный признак — сумма по группам не сходится с числом строк, поэтому её стоит проверять.

</details>

## 6. Сортировка и топ-N

«Покажи три лучших запуска» — это сортировка плюс `head`.

In [ ]:
top = joined.sort_values("val_acc", ascending=False).head(3)   # ascending=False — по убыванию
print(top[["run_id", "model", "val_acc", "family"]])

А теперь посмотрим на другой конец того же порядка — на две последние строки.

In [ ]:
order = runs.sort_values("val_acc", ascending=False)["run_id"].tolist()   # весь порядок от лучшего к худшему
print(order[-2:])   # два последних — казалось бы, худшие запуски

#### ❓ **Вопрос**: Сортировали по убыванию, но в самом конце оказался не худший запуск `r12` (0.771), а `r06`. Почему?

<details>

<summary><strong>Ответ</strong></summary>

`r06` — тот самый запуск без `val_acc`. Пропуски не участвуют в сравнении, и `sort_values` по умолчанию (`na_position="last"`) кладёт их в конец **независимо** от направления сортировки. Поэтому «последняя строка после сортировки по убыванию» — не то же самое, что «худший запуск».

Если нужен именно худший, сначала уберите пропуски: `runs.dropna(subset=["val_acc"])`. Если нужно, чтобы пропуски были видны первыми, — `na_position="first"`.

</details>

Частая задача — «лучшее в каждой группе». Отсортировали по группе и по
метрике, а затем взяли по одной строке из каждой группы.

In [ ]:
by_ds = runs.sort_values(["dataset", "val_acc"], ascending=[True, False])   # сначала по датасету, внутри — по точности вниз
print(by_ds.groupby("dataset").head(1)[["dataset", "run_id", "val_acc"]])   # head(1) внутри группы = лучший в каждой

То же самое одной строкой на один столбец умеет `idxmax`: он возвращает
**метку индекса** строки с максимумом, а не сам максимум.

In [ ]:
print(runs["val_acc"].idxmax())                       # метка строки с максимумом, а не сам максимум
print(runs.loc[runs["val_acc"].idxmax(), "run_id"])   # по метке достаём саму строку

Внутри группы это работает так же: `groupby(...)[столбец].idxmax()` даёт по
одной метке на группу, а `loc` по этим меткам достаёт строки целиком.

In [ ]:
best_idx = runs.groupby("dataset")["val_acc"].idxmax()   # по одной метке на группу
print(best_idx.tolist())
print(runs.loc[best_idx, ["dataset", "run_id", "val_acc"]])   # loc по списку меток — сразу все нужные строки

Ответ тот же, что у `sort_values` + `head(1)` выше, но пути разные:
`idxmax` смотрит на один столбец и возвращает ровно одну строку на группу,
а сортировка позволяет взять топ-N и упорядочить по нескольким ключам.

## 7. Polars: другой язык для той же таблицы

Polars — самостоятельная библиотека (написана на Rust), решающая те же задачи.
Три отличия, ради которых её стоит знать: **выражения** вместо квадратных
скобок, **отсутствие индекса** и **ленивое исполнение**.

In [ ]:
import polars as pl

print(pl.__version__)   # примеры проверены на polars 1.43.2

Читаем тот же самый файл, который писали из pandas: у polars свой `read_csv`.

In [ ]:
pl_runs = pl.read_csv(work / "runs.csv")   # тот же файл, другая библиотека
print(pl_runs.head(3))   # слева нет индекса, зато под именами столбцов напечатаны типы

Уже в выводе видно два отличия: слева нет столбца-индекса, зато прямо под
именами столбцов напечатаны их типы (`str`, `f64`). Отбор строк делает метод
`filter`, отбор столбцов — `select`, а условие внутри — **выражение** `pl.col`.

In [ ]:
print(pl_runs.filter(pl.col("val_acc") > 0.9).select("run_id", "model", "val_acc"))   # filter — строки, select — столбцы

`pl.col("val_acc") > 0.9` само по себе никаких данных не трогает: это
описание вычисления («взять столбец, сравнить»), которое `filter` потом
применяет к своей таблице. Привычная запись из pandas здесь не работает.

In [ ]:
try:
    pl_runs[pl_runs["val_acc"] > 0.9]   # привычная запись из pandas: маска в квадратных скобках
except ValueError as exc:
    print("ValueError:", exc)           # polars ждал список столбцов, а не строк

#### ❓ **Вопрос**: Почему `pl_runs[маска]` не отфильтровал таблицу, а сообщил «expected 7 values ... got 12»?

<details>

<summary><strong>Ответ</strong></summary>

В polars квадратные скобки — это **выбор столбцов**, а не строк. Булев список в них понимается как «оставь те столбцы, где `True`», поэтому polars ждал ровно 7 значений — по числу столбцов — а получил 12, по числу строк.

За этим стоит принципиальное решение: у polars **нет индекса**, строка не имеет метки и адресуется только позицией. Поэтому «оставить строки» — всегда явный `filter(выражение)`, а `.loc`/`.iloc` в polars просто нет. Заодно исчезает и вся история с chained assignment: цепочку `df[...][...] = ...` там негде написать.

</details>

Группировка устроена так же — через выражения. Два новых имени: `pl.len()`
считает строки в группе (аналог `size` в pandas), а `alias` даёт результату
выражения имя — без него столбец назвался бы по самому выражению. Порядок групп
polars не гарантирует, поэтому в конце обычно ставят `sort`.

In [ ]:
agg = pl_runs.group_by("model").agg(
    pl.len().alias("n"),                          # pl.len() — сколько строк в группе (аналог size)
    pl.col("val_acc").mean().alias("mean_acc"),   # alias задаёт имя столбца в результате
)
print(agg.sort("model"))   # порядок групп polars не гарантирует, сортируем сами

### Тот же вопрос в polars: join и пропуски

В разделе 5 мы спрашивали «какая средняя точность у семейств моделей» и
получили молчаливую потерю трёх запусков. Пройдём тот же путь в polars:
`join` вместо `merge`, `fill_null` вместо `fillna`.

Сразу про важное отличие: таблица в polars **неизменяемая**. Написать
`pl_df["family"] = ...`, как в pandas, нельзя — новый столбец не присваивают, а
получают новой таблицей через `with_columns`. Отсюда и присваивания в ячейках
ниже: каждый шаг возвращает новую таблицу, старая остаётся как была.

In [ ]:
pl_models = pl.read_csv(work / "models.csv")                   # тот же справочник, что и в pandas
pl_joined = pl_runs.join(pl_models, on="model", how="left")    # join вместо merge, параметры те же
print(pl_joined["family"].null_count())   # скольким запускам справочник ничего не дал

Три запуска остались без семейства — ровно как в pandas. Дадим им явную категорию.
`fill_null` возвращает **новую** таблицу, поэтому результат надо присвоить обратно.

In [ ]:
pl_joined = pl_joined.with_columns(pl.col("family").fill_null("unknown"))   # with_columns — «таблица с изменённым столбцом»
print(pl_joined["family"].value_counts().sort("family"))   # unknown стал обычной категорией

Все 12 запусков на месте, и те, кому справочник ничего не дал, видны
отдельной категорией `unknown`. Теперь можно считать среднее по семействам.

In [ ]:
print(pl_joined.group_by("family").agg(pl.col("val_acc").mean().round(4)).sort("family"))   # round(4) — чтобы вывод читался

#### ❓ **Вопрос**: В pandas тот же вопрос дал две группы и 8 запусков из 12, а здесь групп три и на месте все 12. Что именно это обеспечило?

<details>

<summary><strong>Ответ</strong></summary>

Две вещи. Первая — `fill_null("unknown")` до группировки: запуски `mobilenet_v3`, которым справочник ничего не дал, получили явную категорию вместо пропуска. Ровно тот же приём в pandas — `fillna("unknown")`.

Вторая — умолчание самого polars: `group_by` **не выбрасывает** строки с пропуском в ключе, без `fill_null` они собрались бы в группу с именем `null`. Это противоположно поведению pandas, где `groupby` их молча теряет, если не написать `dropna=False`.

</details>

### Ленивое исполнение

`read_csv` читает файл сразу. `scan_csv` не читает ничего: он строит **план
запроса**, а данные поедут только по команде `collect`. Это позволяет polars
сначала посмотреть на весь запрос целиком и упростить его.

Немедленное и ленивое исполнение

Слева то, что делает pandas, справа — polars. Посмотрим на правую половину
своими глазами: соберём запрос и напечатаем его план, ничего не выполняя.

In [ ]:
q = (pl.scan_csv(work / "runs.csv")             # scan_csv не читает файл, а начинает план
     .filter(pl.col("val_acc") > 0.9)
     .group_by("model")
     .agg(pl.col("val_acc").max().alias("best")))
print(q.explain())   # explain печатает план: что и в каком порядке polars собрался делать

План напечатан, а файл всё ещё не тронут: `explain` показывает намерение, а не
результат. Просим результат.

In [ ]:
print(q.collect().sort("model"))   # collect — «выполняй»: только здесь файл наконец читается

#### ❓ **Вопрос**: В плане запроса под строкой `Csv SCAN` стоит `PROJECT 2/7 COLUMNS` и `SELECTION: col("val_acc") > 0.9`. Что это значит и почему это важнее на большом файле, чем на нашем?

<details>

<summary><strong>Ответ</strong></summary>

Это перенос отбора **внутрь чтения файла**. Из семи столбцов запросу реально нужны два (`model` и `val_acc`) — остальные читаться не будут (`PROJECT 2/7`). Фильтр по `val_acc` тоже применяется прямо при чтении (`SELECTION`), поэтому в память попадут только подходящие строки.

На таблице из 12 строк это ничего не меняет. На файле в несколько гигабайт разница принципиальная: pandas при `read_csv` сначала целиком поднимет в память все семь столбцов и только потом отбросит ненужное, а ленивый polars может вообще ни разу не прочитать лишние байты. Обратите внимание, что оптимизация стала возможной именно потому, что мы описали **весь** запрос до его выполнения.

</details>

### Словарик: то же самое в pandas и polars

| Задача | pandas | polars |
|---|---|---|
| отобрать строки | `df[маска]` | `df.filter(выражение)` |
| отобрать столбцы | `df[["a", "b"]]` | `df.select("a", "b")` |
| новый столбец | `df["c"] = ...` | `df.with_columns(...)` |
| группировка | `df.groupby("k").agg(...)` | `df.group_by("k").agg(...)` |
| склейка | `df.merge(o, on="k", how="left")` | `df.join(o, on="k", how="left")` |
| пропуски | `fillna` / `isna` / `dropna` | `fill_null` / `is_null` / `drop_nulls` |
| адресация строки | `loc` по метке, `iloc` по позиции | `df.row(i)`, `df.slice(...)`; ни `loc`, ни `iloc` нет |

### Когда что брать

- **pandas** — данные помещаются в память (условно до нескольких гигабайт),
  вокруг вся экосистема: scikit-learn, matplotlib, seaborn, готовые примеры и
  ответы на любой вопрос. Это по-прежнему выбор по умолчанию для исследования.
- **polars** — файл большой или запрос повторяется в конвейере. Выигрыш
  берётся из двух вещей: ленивое исполнение (отбор строк и столбцов уезжает
  внутрь чтения файла) и то, что polars по умолчанию считает **на всех ядрах**
  процессора, тогда как pandas выполняет операцию в одном потоке. Поэтому
  polars часто быстрее и на средних данных, которые в память помещаются.
  Выражения сложнее прочитать, но труднее написать неверно (нет индекса — нет
  тихих ошибок с `loc`).

Таблицы конвертируются друг в друга: `pl.from_pandas(df)` и `pl_df.to_pandas()`.
Обеим нужен установленный `pyarrow` — проверим, есть ли он.

In [ ]:
try:
    print(type(pl_runs.to_pandas()).__name__)   # polars → pandas
    print(pl.from_pandas(models).shape)         # pandas → polars, форма сохранилась
except ImportError as exc:
    print("нужен отдельный пакет:", exc)        # мост между библиотеками требует pyarrow

Универсальный запасной мост между любыми инструментами — файл. Csv для
этого плох: типы столбцов в нём не хранятся и при каждом чтении угадываются
заново. `Parquet` — двоичный формат, который помнит типы и который читают обе
библиотеки. Он **колоночный**: значения одного столбца лежат подряд,
поэтому их хорошо сжимать и можно прочитать только нужные столбцы, не
трогая остальные. Это ровно та экономия, которую мы видели в плане запроса
polars как `PROJECT 2/7 COLUMNS`, только на уровне самого файла.

In [ ]:
pl_runs.write_parquet(work / "runs.parquet")            # двоичный формат, помнит типы столбцов
print(pl.read_parquet(work / "runs.parquet").shape)     # прочитали целиком
print(pl.read_parquet(work / "runs.parquet", columns=["run_id", "val_acc"]).shape)   # только два столбца — остальные с диска не поднимались

Вторая строка — то самое чтение «только нужных столбцов»: два из семи,
остальные пять с диска даже не поднимались. У pandas всё то же самое и с
теми же именами параметров, только через `pyarrow`.

In [ ]:
try:
    runs.to_parquet(work / "from_pandas.parquet")   # у pandas те же имена методов
    back = pd.read_parquet(work / "from_pandas.parquet", columns=["run_id", "val_acc"])
    print(back.shape)
    print(back.dtypes.equals(runs[["run_id", "val_acc"]].dtypes))   # типы приехали из файла, а не угаданы заново
except ImportError as exc:
    print("для pandas нужен pyarrow:", exc)

`dtypes.equals` вернул `True`: типы столбцов приехали из файла, а не были
угаданы заново, как это происходит с csv. И читать по столбцам csv не даёт —
там строка файла это строка таблицы, чтобы добраться до второго столбца,
придётся разобрать все. Своего parquet у polars хватает без дополнительных
пакетов, pandas просит тот же `pyarrow`.

## Итог

- `DataFrame` — таблица со столбцами и индексом, `Series` — один столбец.
  Тип столбца один на весь столбец; в pandas 3 текст — это `str`, а не `object`.
- Индекс — не нумерация, а ключ: по нему работает `loc` и по нему pandas
  выравнивает данные в арифметике и при склейке.
- `read_csv`/`to_csv`: не забываем `index=False`, иначе получаем `Unnamed: 0`.
  Первое, что делают с таблицей: `shape`, `head`, `dtypes`, `describe`.
- Строки отбирают маской, `loc` работает по метке, `iloc` — по позиции; после
  фильтрации это разные вещи.
- Писать в таблицу — только `df.loc[строки, столбец] = ...`. Присваивание через
  цепочку в pandas 3 не работает никогда.
- `NaN` — «неизвестно», не ноль. `fillna(0)` меняет ответ; `dropna()` без
  `subset` выбрасывает лишнее; `groupby` тихо теряет строки с пропуском в ключе.
- `groupby` + `agg` — та же модель, что `GROUP BY` в SQL; `merge` — та же, что
  `JOIN`, и `how="left"` безопаснее `inner`.
- Polars — выражения вместо скобок, нет индекса, `join`/`fill_null` вместо
  `merge`/`fillna`, `with_columns` вместо присваивания столбца. `scan_csv` +
  `collect` дают ленивое исполнение с отбором столбцов и строк прямо при
  чтении, а считает polars на всех ядрах сразу.
- Передавать таблицу между инструментами лучше через `parquet`: он помнит
  типы столбцов, csv их каждый раз угадывает заново.

Задачи — в `tasks.md`; данные готовит `python3 assets/make_data.py ~/seminar-13`,
работаем в том же домашнем каталоге `~/seminar-13/`, что и демо. Домашка —
мини-исследование датасета в pandas.